In [1]:
import sys
import json
from pathlib import Path
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

sys.path.insert(0, "src")

from sugar_jepa.sugar_jepa_model import JepaEncoder
from common.evaluation.device import resolve_torch_device

device = torch.device(resolve_torch_device())
print(f"using {device}")

DATA = Path("data/input/loop_ai_ready_joined2.csv")
JEPA_RUNS = Path("jepa_nudes")
WINDOW_STRIDE = 4

/Users/neprog/Desktop/kse/audio_dl/glucose-forecasting/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


using mps


In [2]:
JEPA_ENCODER_ARGS = {"n_time_steps", "patch_size", "embed_dim", "n_layers", "n_heads", "mlp_ratio", "dropout", "norm"}

def load_jepa_encoder(run_dir, checkpoint="encoder.pt"):
    run_dir = Path(run_dir)
    config = json.loads((run_dir / "config.json").read_text())
    kwargs = {k: v for k, v in config.items() if k in JEPA_ENCODER_ARGS}
    encoder = JepaEncoder(**kwargs)
    state_dict = torch.load(run_dir / checkpoint, map_location=device)
    encoder.load_state_dict(state_dict)
    return encoder.to(device).eval(), config["n_time_steps"]

encoder_288, WINDOW_288 = load_jepa_encoder(
    JEPA_RUNS / "jepa_encoder-288" / "jepa_encoder_w288_p8_d96_l3_h6_20260813_000957"
)
encoder_864, WINDOW_864 = load_jepa_encoder(
    JEPA_RUNS / "jepa_encoder-864" / "jepa_encoder_w864_p8_d96_l3_h6_20260820_002218"
)
print(f"encoder_288: window={WINDOW_288}, embed_dim={encoder_288.embed_dim}")
print(f"encoder_864: window={WINDOW_864}, embed_dim={encoder_864.embed_dim}")

encoder_288: window=288, embed_dim=96
encoder_864: window=864, embed_dim=96


In [3]:
DATA_COLS = ["sequence_id", "Timestamp", "Glucose (mg/dL)", "User ID", "Study Group", "Recommended Split"]

df = pd.read_csv(DATA, usecols=DATA_COLS, parse_dates=["Timestamp"])
df = df.dropna(subset=["Glucose (mg/dL)"])
df = df.sort_values(["sequence_id", "Timestamp"])
df = df[df["Recommended Split"] == "test"]

print(f"{len(df):,} rows, {df['sequence_id'].nunique():,} sequences")

1,860,222 rows, 1,392 sequences


In [4]:
class RawWindowDataset(Dataset):
    def __init__(
        self,
        df,
        group_ids,
        user_to_label,
        window_steps,
        seq_col="User ID",
        glucose_col="Glucose (mg/dL)",
        user_col="User ID",
        stride=WINDOW_STRIDE,
    ):
        self.window_steps = window_steps
        self.user_to_label = user_to_label

        self.series, self.users = [], []
        self.index = []  # (series_idx, start)

        kept = df[df[seq_col].isin(set(group_ids))]
        n_skipped = 0
        for gid, g in kept.groupby(seq_col, sort=False):
            values = g[glucose_col].to_numpy(dtype=np.float32)
            n_windows = len(values) - window_steps + 1
            if n_windows <= 0:
                n_skipped += 1
                continue
            si = len(self.series)
            self.series.append(values)
            self.users.append(g[user_col].iloc[0])
            for start in range(0, n_windows, stride):
                self.index.append((si, start))

        if n_skipped:
            print(f"skipped {n_skipped} groups shorter than {window_steps} steps")

    def __len__(self):
        return len(self.index)

    def __getitem__(self, idx):
        si, start = self.index[idx]
        window = self.series[si][start : start + self.window_steps]
        label = self.user_to_label[self.users[si]]
        return torch.from_numpy(window.copy()), label

In [5]:
TEST_FRACTION = 0.2

def split_rows_per_user(df, user_col="User ID", ts_col="Timestamp", test_frac=TEST_FRACTION, window_steps=WINDOW_864):
    train_parts, test_parts = [], []
    n_skipped = 0
    for _, g in df.groupby(user_col, sort=False):
        g = g.sort_values(ts_col)
        if len(g) < 2 * window_steps:
            n_skipped += 1
            continue
        cut = int(len(g) * (1 - test_frac))
        cut = max(window_steps, min(cut, len(g) - window_steps))
        train_parts.append(g.iloc[:cut])
        test_parts.append(g.iloc[cut:])
    if n_skipped:
        print(f"skipped {n_skipped} users (fewer than {2 * window_steps} rows total)")
    return pd.concat(train_parts), pd.concat(test_parts)

train_df_288, test_df_288 = split_rows_per_user(df, window_steps=WINDOW_288)
ALL_USERS_288 = sorted(pd.concat([train_df_288, test_df_288])["User ID"].unique())
USER_TO_LABEL_288 = {user: i for i, user in enumerate(ALL_USERS_288)}
NUM_CLASSES_288 = len(USER_TO_LABEL_288)
print(f"288-window: {NUM_CLASSES_288} users")

train_df_864, test_df_864 = split_rows_per_user(df, window_steps=WINDOW_864)
ALL_USERS_864 = sorted(pd.concat([train_df_864, test_df_864])["User ID"].unique())
USER_TO_LABEL_864 = {user: i for i, user in enumerate(ALL_USERS_864)}
NUM_CLASSES_864 = len(USER_TO_LABEL_864)
print(f"864-window: {NUM_CLASSES_864} users")

train_ds_288 = RawWindowDataset(train_df_288, train_df_288["User ID"].unique(), USER_TO_LABEL_288, window_steps=WINDOW_288)
test_ds_288 = RawWindowDataset(test_df_288, test_df_288["User ID"].unique(), USER_TO_LABEL_288, window_steps=WINDOW_288)

train_ds_864 = RawWindowDataset(train_df_864, train_df_864["User ID"].unique(), USER_TO_LABEL_864, window_steps=WINDOW_864)
test_ds_864 = RawWindowDataset(test_df_864, test_df_864["User ID"].unique(), USER_TO_LABEL_864, window_steps=WINDOW_864)

skipped 3 users (fewer than 576 rows total)
288-window: 351 users
skipped 17 users (fewer than 1728 rows total)
864-window: 337 users


In [6]:
BATCH_SIZE = 256

train_dl_288 = DataLoader(train_ds_288, batch_size=BATCH_SIZE, shuffle=True)
test_dl_288 = DataLoader(test_ds_288, batch_size=BATCH_SIZE)

train_dl_864 = DataLoader(train_ds_864, batch_size=BATCH_SIZE, shuffle=True)
test_dl_864 = DataLoader(test_ds_864, batch_size=BATCH_SIZE)

In [7]:
class JepaUserProbe(nn.Module):
    def __init__(self, encoder, embed_dim, num_classes):
        super().__init__()
        self.encoder = encoder
        self.encoder.requires_grad_(False)
        self.encoder.eval()
        self.head = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        with torch.no_grad():
            z = self.encoder(x)
        pooled = z.mean(dim=1)
        return self.head(pooled)

In [8]:
import lightning as L

class LightningProbe(L.LightningModule):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def _step(self, batch, stage):
        x, y = batch
        logits = self.model(x)
        loss = F.cross_entropy(logits, y)
        acc = (logits.argmax(dim=-1) == y).float().mean()
        self.log(f"{stage}_loss", loss, prog_bar=True, on_epoch=True)
        self.log(f"{stage}_acc", acc, prog_bar=True, on_epoch=True)
        return loss

    def training_step(self, batch, batch_idx):
        return self._step(batch, "train")

    def validation_step(self, batch, batch_idx):
        return self._step(batch, "val")

    def configure_optimizers(self):
        return torch.optim.Adam(self.model.head.parameters(), lr=1e-3)

In [ ]:
from lightning.pytorch.loggers import CSVLogger

logger_288 = CSVLogger(save_dir="runs/probe_logs", name="w288")

probe_288 = JepaUserProbe(encoder_288, embed_dim=encoder_288.embed_dim, num_classes=NUM_CLASSES_288)
lit_288 = LightningProbe(probe_288)

trainer_288 = L.Trainer(
    max_epochs=15,
    enable_checkpointing=False,
    logger=logger_288,
)
trainer_288.fit(lit_288, train_dl_288, test_dl_288)

In [ ]:
logger_864 = CSVLogger(save_dir="runs/probe_logs", name="w864")

probe_864 = JepaUserProbe(encoder_864, embed_dim=encoder_864.embed_dim, num_classes=NUM_CLASSES_864)
lit_864 = LightningProbe(probe_864)

trainer_864 = L.Trainer(
    max_epochs=15,
    enable_checkpointing=False,
    logger=logger_864,
)
trainer_864.fit(lit_864, train_dl_864, test_dl_864)

In [ ]:

SAVE_DIR = Path("runs/probe_heads")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

torch.save(probe_288.head.state_dict(), SAVE_DIR / "w288_head.pt")
torch.save(probe_864.head.state_dict(), SAVE_DIR / "w864_head.pt")

print(f"saved heads to {SAVE_DIR}")

In [ ]:
import matplotlib.pyplot as plt

def epoch_series(csv_path, col):
    m = pd.read_csv(csv_path)
    m = m.dropna(subset=[col])
    return m["epoch"], m[col]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for name, log_dir in [("w288", logger_288.log_dir), ("w864", logger_864.log_dir)]:
    csv_path = Path(log_dir) / "metrics.csv"

    ep, val = epoch_series(csv_path, "train_loss_epoch")
    axes[0].plot(ep, val, label=f"{name} train")
    ep, val = epoch_series(csv_path, "val_loss")
    axes[0].plot(ep, val, linestyle="--", label=f"{name} val")

    ep, val = epoch_series(csv_path, "train_acc_epoch")
    axes[1].plot(ep, val, label=f"{name} train")
    ep, val = epoch_series(csv_path, "val_acc")
    axes[1].plot(ep, val, linestyle="--", label=f"{name} val")

axes[0].set_title("Loss"); axes[0].set_xlabel("epoch"); axes[0].legend()
axes[1].set_title(f"Accuracy (w288: {NUM_CLASSES_288} classes, w864: {NUM_CLASSES_864} classes)")
axes[1].set_xlabel("epoch"); axes[1].legend()
plt.tight_layout()
plt.show()